# Tensores

### 1. Qué es un Tensor
- Estructura similar a las matrices y a las arrays de Numpy por lo tanto son compatibles
- Pueden correr en CPU o GPU

In [ ]:
import torch
import numpy as np

### Como inicializar un Tensor

Se pueden inicializar de variaas maneras

**Desde datos directamente**
- A partir de listas o valores
- El tipo de dato dtype se infiere automaticamente

In [ ]:
data = [[1, 2],[3, 4]]
x_data = torch.tensor(data)

**Desde un array de Numpy**
- se puede crear un tensor a partir de ndarray
- se puede convertir un tensor a numpy directamente

In [ ]:
np_array = np.array(data)
x_np = torch.from_numpy(np_array)

**Desde otro tensor**

- El nuevo tensor hereda propiedades del tensor original: forma, tipo de dato.

- Se pueden sobrescribir explícitamente si se desea.

In [ ]:
x_ones = torch.ones_like(x_data) # retiene las propiedades de x_data
print(f"Ones Tensor: \n {x_ones} \n")

x_rand = torch.rand_like(x_data, dtype=torch.float) # sobreescribe el tipo de dato de x_data
print(f"Random Tensor: \n {x_rand} \n")

**Inicialización con valores aleatorios o constantes**

Se pueden crear tensores con ceros, unos o valores aleatorios, especificando la forma (shape) como una tupla.

Ejemplos:

In [ ]:
shape = (2, 3)  # 2 filas, 3 columnas

zeros = torch.zeros(shape)       # todos ceros
ones = torch.ones(shape)         # todos unos
rand = torch.rand(shape)         # valores aleatorios entre 0 y 1
randn = torch.randn(shape)       # valores aleatorios con distribución normal (media=0, var=1)
full = torch.full(shape, 7)      # todos con valor 7


### Atributos de un Tensor

- x.shape → devuelve las dimensiones del tensor

- x.size() → lo mismo que x.shape

- x.dtype → tipo de datos (torch.float32, torch.int64, etc.)

- x.device → dispositivo donde está el tensor (cpu o cuda)

- x.requires_grad → si el tensor está marcado para calcular gradientes

Ejemplo:

In [ ]:
x = torch.rand(2, 3, dtype=torch.float32, device='cuda', requires_grad=True)
print(x.shape)         # torch.Size([2, 3])
print(x.dtype)         # torch.float32
print(x.device)        # cuda:0
print(x.requires_grad) # True

### Operaciones avanzadas con Tensores
- Se puede hacer con @ o matmul().
- También se puede guardar el resultado en un tensor preexistente usando out=.

In [ ]:
y1 = tensor @ tensor.T         # operador @
y2 = tensor.matmul(tensor.T)   # método matmul
y3 = torch.rand_like(y1)       
torch.matmul(tensor, tensor.T, out=y3)  # resultado guardado en y3

Producto elemento a elemento (Hadamard product)

- Se puede hacer con * o mul().

- También admite out= para escribir el resultado en un tensor existente.

In [ ]:
z1 = tensor * tensor
z2 = tensor.mul(tensor)
z3 = torch.rand_like(tensor)
torch.mul(tensor, tensor, out=z3)

Agregación y conversión a valor de Python

- Para sumar todos los elementos y obtener un número de Python:

In [ ]:
agg = tensor.sum()
agg_item = agg.item()  # convierte a float/int
print(agg_item, type(agg_item))

Operaciones in-place

- Modifican el tensor original directamente (nota el _ al final del método):

In [ ]:
tensor.add_(5)  # suma 5 a todos los elementos de tensor en su lugar

# Datasets

### Creacion de un Dataset a partir de datos

- Para crear un dataset propio, tu clase debe heredar de torch.utils.data.Dataset y definir tres métodos clave:

__init__(self, ...)

- Se ejecuta al crear el objeto.

- Sirve para cargar rutas de archivos, etiquetas, transformaciones, etc.

__len__(self)

- Devuelve el número total de muestras en el dataset.

- Permite que PyTorch sepa cuántos elementos hay.

__getitem__(self, idx)

- Devuelve la muestra y su etiqueta en la posición idx.

Aquí se pueden aplicar transformaciones o preprocesamiento a la muestra.

In [ ]:
import os
import pandas as pd
from torchvision.io import decode_image

class CustomImageDataset(Dataset):
    def __init__(self, annotations_file, img_dir, transform=None, target_transform=None):
        self.img_labels = pd.read_csv(annotations_file)
        self.img_dir = img_dir
        self.transform = transform
        self.target_transform = target_transform

    def __len__(self):
        return len(self.img_labels)

    def __getitem__(self, idx):
        img_path = os.path.join(self.img_dir, self.img_labels.iloc[idx, 0])
        image = decode_image(img_path)
        label = self.img_labels.iloc[idx, 1]
        if self.transform:
            image = self.transform(image)
        if self.target_transform:
            label = self.target_transform(label)
        return image, label

### Preparar lso datos para entrenar con DataLoaders

- Un Dataset devuelve una muestra a la vez (feature y etiqueta).

- Para entrenamiento, necesitamos:

- Minibatches: pasar varias muestras a la vez para eficiencia y estabilidad en el entrenamiento.

- Barajar los datos (shuffle): evita que el modelo aprenda el orden de los datos y reduce overfitting.

- Multiprocesamiento (num_workers): acelera la carga de datos usando varios procesos de Python.

- DataLoader abstrae toda esta complejidad y permite acceder a los datos fácilmente en un bucle de entrenamiento.

In [ ]:
from torch.utils.data import DataLoader

train_dataloader = DataLoader(training_data, batch_size=64, shuffle=True)
test_dataloader = DataLoader(test_data, batch_size=64, shuffle=True)

### Iterar a traves del DataLoader
- Cada iteración devuelve un minibatch de features y labels.

- El tamaño del minibatch se define con batch_size al crear el DataLoader.

- Si shuffle=True, los datos se barajan automáticamente al completar una pasada (epoch).

- Para control más avanzado del orden de carga, se pueden usar Samplers.

In [ ]:
# Display image and label.
train_features, train_labels = next(iter(train_dataloader))
print(f"Feature batch shape: {train_features.size()}")
print(f"Labels batch shape: {train_labels.size()}")
img = train_features[0].squeeze()
label = train_labels[0]
plt.imshow(img, cmap="gray")
plt.show()
print(f"Label: {label}")

# Transforms
- Los datos no siempre vienen en el formato que necesitan los modelos.

- Transforms permiten modificar features y labels para que sean adecuados para entrenamiento.

- Parámetros clave en datasets de TorchVision

- transform → transforma las features (ej. imágenes)

- target_transform → transforma las labels (ej. one-hot encoding)

- Transformaciones comunes

- Usando el módulo torchvision.transforms:

- ToTensor → convierte imágenes PIL o NumPy a tensores.

- Lambda → aplica cualquier función personalizada sobre features o labels.

- Se pueden encadenar varias transformaciones con transforms.Compose([...]).

In [ ]:
import torch
from torchvision import datasets
from torchvision.transforms import ToTensor, Lambda

ds = datasets.FashionMNIST(
    root="data",
    train=True,
    download=True,
    transform=ToTensor(),
    target_transform=Lambda(lambda y: torch.zeros(10, dtype=torch.float).scatter_(0, torch.tensor(y), value=1))
)

### Ejemplo con lambda

In [ ]:
target_transform = Lambda(lambda y: torch.zeros(
    10, dtype=torch.float).scatter_(dim=0, index=torch.tensor(y), value=1))